# Multi-config halo-mass posterior sweeps

This notebook is a compact follow-up to `run_simulations.ipynb`.
It builds the analogue catalogue once, then reruns the posterior inference for several feature configurations such as separation only, separation plus velocities, and separation plus velocities plus stellar masses.

Use the settings cell below to choose:
- `PAIR_SUBSET`: `"blue_blue"`, `"red_red"`, `"blue_or_red"`, or `"all"`
- `INFERENCE_METHOD`: `"abc"`, `"gmm"`, or `"sbi"`

The active target is still read from `config/config.json`, and this notebook expects that target to be `total_virial_mass`.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
repo_root = next((path for path in [cwd, *cwd.parents] if (path / "analogues").is_dir()), None)
if repo_root is None:
    raise FileNotFoundError("Could not find the repo root containing the `analogues` package.")

for pth in [repo_root, repo_root / "scripts", repo_root / "illustris_python"]:
    if pth.exists() and str(pth) not in sys.path:
        sys.path.insert(0, str(pth))

from config.loader import load_config
from lg_multi_config_posteriors import (
    DEFAULT_SBI_CONFIG,
    build_analogue_sample,
    build_pair_catalog,
    pair_subset_display_name,
    plot_configuration_posteriors,
    run_configuration_grid,
)

print("Repo root:", repo_root)


## Settings

You do not need to toggle feature `active` flags in `config/config.json` for this notebook.
The config file is used for the observed values, sigmas, selection/filter settings, and target metadata, while the feature subsets are defined explicitly below.


In [ ]:
config_loader = load_config()
analysis_config = config_loader.get_config()

if analysis_config.target_name != "total_virial_mass":
    raise ValueError(
        "This notebook expects `total_virial_mass` to be the active target in config/config.json. "
        f"Found: {analysis_config.target_name!r}"
    )

FEATURE_INFO = analysis_config.all_features
LOG10_FEATURES = {
    name for name, meta in FEATURE_INFO.items()
    if bool(meta.get("log10", False))
}

TARGET_COLUMN = analysis_config.target["column"]
TARGET_LABEL = analysis_config.target.get("label", TARGET_COLUMN)
TARGET_LOG10 = bool(analysis_config.target.get("log10", False))

PAIR_SUBSET = "blue_blue"
INFERENCE_METHOD = "abc"  # choose from: "abc", "gmm", "sbi"
SNAP = 99

FEATURE_CONFIGS = [
    {"label": "Separation only", "features": ["r_kpc"]},
    {"label": r"$v_r$ + sep", "features": ["r_kpc", "v_r"]},
    {"label": r"$v_r$ + $v_t$ + sep", "features": ["r_kpc", "v_r", "v_t"]},
    {
        "label": r"$v_r$ + $v_t$ + sep + $M_*$",
        "features": ["r_kpc", "v_r", "v_t", "mstar_big", "mstar_small"],
    },
]

ABC_ACCEPT_FRAC = 0.05
ABC_KERNEL_BANDWIDTH = 1.0

GMM_COMPONENTS = "bic"
GMM_RANDOM_STATE = 42

NUM_POSTERIOR_SAMPLES = 20_000
RANDOM_STATE = 42
SBI_CONFIG = dict(DEFAULT_SBI_CONFIG)

REFERENCE_LOG_MASS = 12.6
REFERENCE_LABEL = "Real LG estimate ~12.6 (van der Marel 2012)"
PLOT_XLABEL = r"log total virial mass [$M_{\odot}$]"

feature_rows = []
for model_config in FEATURE_CONFIGS:
    for feature_name in model_config["features"]:
        meta = FEATURE_INFO[feature_name]
        feature_rows.append(
            {
                "model": model_config["label"],
                "feature": feature_name,
                "observed_value": meta["obs"],
                "sigma": meta["sigma"],
                "label": meta.get("label", feature_name),
                "description": meta.get("description", ""),
            }
        )

display(pd.DataFrame(feature_rows))
display(config_loader.show_active_target())
print("Pair subset:", pair_subset_display_name(PAIR_SUBSET))
print("Inference method:", INFERENCE_METHOD)


## Build the analogue catalogue once

This follows the same selection and filtering logic as `run_simulations.ipynb`, but the pair subset is applied only once and then reused by every feature configuration.


In [ ]:
analogue_sample = build_analogue_sample(
    repo_root,
    selection_config=analysis_config.selection,
    filter_config=analysis_config.filter,
    snap=SNAP,
    verbose=True,
)

display(pd.DataFrame(analogue_sample._pipeline.get_cutflow()))

catalog, color_summary = build_pair_catalog(
    analogue_sample,
    pair_subset=PAIR_SUBSET,
)

display(color_summary)


## Run the feature sweep

Each row below corresponds to one posterior curve in the final figure.


In [ ]:
results, summary_df = run_configuration_grid(
    catalog=catalog,
    feature_info=FEATURE_INFO,
    configurations=FEATURE_CONFIGS,
    target_column=TARGET_COLUMN,
    target_label=TARGET_LABEL,
    method=INFERENCE_METHOD,
    log10_features=LOG10_FEATURES,
    target_log10=TARGET_LOG10,
    abc_accept_frac=ABC_ACCEPT_FRAC,
    abc_kernel_bandwidth=ABC_KERNEL_BANDWIDTH,
    gmm_components=GMM_COMPONENTS,
    gmm_random_state=GMM_RANDOM_STATE,
    sbi_config=SBI_CONFIG,
    num_samples=NUM_POSTERIOR_SAMPLES,
    random_state=RANDOM_STATE,
)

display(summary_df)


## Plot the posteriors

The dashed vertical line is optional and is included here to match the comparison style in your example figure.


In [ ]:
title = f"{pair_subset_display_name(PAIR_SUBSET)}: LG Halo Mass Posterior - All Models"

fig, ax = plot_configuration_posteriors(
    results,
    title=title,
    x_label=PLOT_XLABEL,
    reference_value=REFERENCE_LOG_MASS,
    reference_label=REFERENCE_LABEL,
    figsize=(10, 6),
)
ax.set_xlim(11.0, 15.0)
fig
